In [3]:
import json
import os
import re
import nltk

# Ensure NLTK punkt is available
try:
    nltk.data.find("tokenizers/punkt")
except LookupError:
    nltk.download("punkt")

# -----------------------------
# Khmer Text Preprocessing Utils
# -----------------------------

def clean_khmer_text(text: str) -> str:
    if not text:
        return ""
    text = re.sub(r"[^\u1780-\u17FF0-9\u17E0-\u17E9\s។៕៖,]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize_khmer_words(text: str):
    rough_tokens = text.split()
    tokens = []
    for token in rough_tokens:
        if len(token) > 20:
            tokens.extend(list(token))
        else:
            tokens.append(token)
    return tokens

def count_sentences_khmer(text: str):
    sentences = re.split(r"[។៕]+", text)
    sentences = [s.strip() for s in sentences if s.strip()]
    return len(sentences)

def extract_tags_from_title(title: str):
    tokens = tokenize_khmer_words(title)
    tags = [t for t in tokens if len(t) >= 4]
    return list(dict.fromkeys(tags))

def preprocess_article(json_data: dict):
    title = clean_khmer_text(json_data.get("title", ""))
    content = clean_khmer_text(json_data.get("content", ""))
    tags = extract_tags_from_title(title)
    words = tokenize_khmer_words(content)
    word_count = len(words)
    sentence_count = count_sentences_khmer(content)
    character_count = len(content.replace(" ", ""))
    return {
        "url": json_data.get("url", ""),
        "source": json_data.get("source", ""),
        "publication_date": json_data.get("publication_date", ""),
        "scrape_date": json_data.get("scrape_date", ""),
        "title": title,
        "content": content,
        "tags": tags,
        "word_count": word_count,
        "sentence_count": sentence_count,
        "character_count": character_count
    }

# -----------------------------
# Batch Process Function (Single JSON Output)
# -----------------------------

def process_json_file(input_path: str, output_path: str):
    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    processed_articles = []

    # Support list of articles
    if isinstance(data, list):
        for article in data:
            processed_articles.append(preprocess_article(article))
    elif isinstance(data, dict):
        processed_articles.append(preprocess_article(data))
    else:
        print("❌ Unsupported JSON format!")
        return

    # Ensure output folder exists
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    # Save all articles in a single JSON file
    with open(output_path, "w", encoding="utf-8") as out:
        json.dump(processed_articles, out, ensure_ascii=False, indent=2)

    print(f"✔ All articles saved in single JSON: {output_path}")


if __name__ == "__main__":
    INPUT_FILE = r"D:\CADT\CapstoneProjectII\Haksou\dab_news.json"
    OUTPUT_FILE = r"D:\CADT\CapstoneProjectII\Haksou\all_articles_preprocessed.json"

    if os.path.exists(INPUT_FILE):
        process_json_file(INPUT_FILE, OUTPUT_FILE)
    else:
        print("❌ JSON input file not found!")


❌ JSON input file not found!


In [4]:
import json
import os
import re
import nltk
from typing import Dict, List

# Ensure NLTK punkt is available
try:
    nltk.data.find("tokenizers/punkt")
except LookupError:
    nltk.download("punkt")


# -----------------------------
# Khmer Text Preprocessing Utils
# -----------------------------

def clean_khmer_text(text: str) -> str:
    if not text:
        return ""
    # Keep Khmer letters, Khmer numbers, Latin numbers, Khmer punctuation
    text = re.sub(r"[^\u1780-\u17FF0-9\u17E0-\u17E9\s។៕៖,]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize_khmer_words(text: str):
    rough_tokens = text.split()
    tokens = []
    for token in rough_tokens:
        if len(token) > 20:
            tokens.extend(list(token))  # fallback for long glued words
        else:
            tokens.append(token)
    return tokens


def count_sentences_khmer(text: str):
    sentences = re.split(r"[។៕]+", text)
    sentences = [s.strip() for s in sentences if s.strip()]
    return len(sentences)


# -----------------------------
# Category-Based Tags
# -----------------------------

categories: Dict[str, List[str]] = {
    'agriculture': ['កសិកម្ម', 'ស្រូវ', 'ដំណាំ', 'កសិករ', 'ដីស្រែ', 'ជី', 'សត្វចិញ្ចឹម', 'ទឹកស្រោចស្រព', 'ទីផ្សារកសិផល', 'គ្រាប់ពូជ'],
    'public': ['សាធារណៈ', 'រដ្ឋ', 'សេវាសាធារណៈ', 'អាជ្ញាធរ', 'សង្គម'],
    'commerce': ['ពាណិជ្ជកម្ម', 'អាជីវកម្ម', 'ទីផ្សារ', 'ពាណិជ្ជកម្មអន្តរជាតិ', 'ការនាំចេញ', 'ការនាំចូល'],
    'religion': ['សាសនា', 'ព្រះ', 'វត្ត', 'សាសនាចក្រ', 'សមាគមសាសនា'],
    'culture': ['វប្បធម៌', 'ប្រពៃណី', 'សិល្បៈ', 'តន្ត្រី', 'រាំ', 'ភាពយន្ត', 'បុរាណ', 'ពិធីបុណ្យ'],
    'economy': ['សេដ្ឋកិច្ច', 'ធនាគារ', 'វិនិយោគ', 'ទីផ្សារ', 'ការងារ', 'ប្រាក់ខែ', 'អតិផរណា'],
    'education': ['អប់រំ', 'សាលា', 'សិក្សា', 'គ្រូ', 'និស្សិត', 'មហាវិទ្យាល័យ', 'សាកលវិទ្យាល័យ'],
    'sports': ['កីឡា', 'បាល់ទាត់', 'វាយកូនបាល់', 'អត្តពលិក', 'ការប្រកួត', 'អូឡាំពិក', 'កីឡាជាតិ'],
    'environment': ['បរិស្ថាន', 'ធម្មជាតិ', 'ទន្លេ', 'ព្រៃឈើ', 'អាកាសធាតុ', 'ការបំពុល', 'សត្វព្រៃ'],
    'international_news': ['ពត៌មានអន្តរជាតិ', 'អន្តរជាតិ', 'ពិភពលោក', 'កិច្ចប្រជុំអន្តរជាតិ', 'អង្គការសហប្រជាជាតិ'],
    'health': ['សុខាភិបាល', 'ពេទ្យ', 'មន្ទីរពេទ្យ', 'ជំងឺ', 'ថ្នាំ', 'វេជ្ជសាស្ត្រ', 'វ៉ាក់សាំង', 'អនាម័យ'],
    'industry': ['ឧស្សាហកម្ម', 'រោងចក្រ', 'ផលិតកម្ម', 'ឧស្សាហកម្មផលិតផល', 'សេវាឧស្សាហកម្ម'],
    'technology': ['បច្ចេកវិទ្យា', 'កុំព្យូទ័រ', 'អ៊ិនធឺណេត', 'ឌីជីថល', 'កម្មវិធី', 'ទិន្នន័យ', 'ហេដ្ឋារចនាសម្ព័ន្ធ'],
    'national_news': ['ពត៌មានជាតិ', 'សារព័ត៌មានជាតិ', 'ប្រទេស', 'រាជរដ្ឋាភិបាល', 'សភា', 'អាជ្ញាធរ'],
    'justice': ['យុត្តិធម៍', 'ច្បាប់', 'សាលា', 'ប៉ូលិស', 'អង្គការយុត្តិធម៍'],
    'labor': ['ការងារ', 'បុគ្គលិក', 'ប្រាក់ខែ', 'សិទ្ធិកម្មករ', 'សហភាព'],
    'conflict_war': ['សង្រ្គាម', 'ជម្លោះ', 'ការប៉ះទង្គិច', 'កងទ័ព', 'អាវុធ', 'ការវាយប្រហារ'],
    'telecommunication': ['ទូរគមនាគមន៍', 'ទូរស័ព្ទ', 'អ៊ិនធឺណេត', 'សញ្ញា', 'ខ្សែទូរស័ព្ទ'],
    'traffic_accident': ['គ្រោះថ្នាក់ចរាចរណ៍', 'បុកគ្នា', 'ឡានបុក', 'ម៉ូតូបុក', 'អ្នករបួស', 'ស្លាប់'],
    'tourism': ['ទេសចរណ៍', 'អង្គរវត្ត', 'សៀមរាប', 'ឆ្នេរ', 'សណ្ឋាគារ', 'មគ្គុទ្ទេសក៍', 'ប្រាសាទ'],
    'aviation': ['អាកាសចរណ៍', 'អាកាសយាន', 'យន្តហោះ', 'កំពង់ផែ', 'ហោះឆ្នេរ']
}


def extract_tags_and_category(title: str, content: str):
    text = title + " " + content
    tags = set()
    category_count = {cat: 0 for cat in categories}

    for cat, keywords in categories.items():
        for kw in keywords:
            if kw in text:
                tags.add(kw)
                category_count[cat] += 1

    # Determine primary category
    primary_category = None
    category_confidence = 0.0
    if category_count:
        sorted_cats = sorted(category_count.items(), key=lambda x: x[1], reverse=True)
        if sorted_cats[0][1] > 0:
            primary_category = sorted_cats[0][0]
            total_matches = sum(category_count.values())
            category_confidence = sorted_cats[0][1] / total_matches if total_matches > 0 else 0.0

    return list(tags), primary_category, category_confidence


# -----------------------------
# Main Preprocessor Function
# -----------------------------

def preprocess_article(json_data: dict):
    title = clean_khmer_text(json_data.get("title", ""))
    content = clean_khmer_text(json_data.get("content", ""))

    tags, primary_category, category_confidence = extract_tags_and_category(title, content)

    words = tokenize_khmer_words(content)
    word_count = len(words)
    sentence_count = count_sentences_khmer(content)
    character_count = len(content.replace(" ", ""))

    return {
        "url": json_data.get("url", ""),
        "source": json_data.get("source", ""),
        "publication_date": json_data.get("publication_date", ""),
        "scrape_date": json_data.get("scrape_date", ""),
        "title": title,
        "content": content,
        "tags": tags,
        "word_count": word_count,
        "sentence_count": sentence_count,
        "character_count": character_count,
        "primary_category": primary_category,
        "category_confidence": category_confidence
    }


# -----------------------------
# Batch Process Function (Single JSON Output)
# -----------------------------

def process_json_file(input_path: str, output_path: str):
    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    processed_articles = []

    if isinstance(data, list):
        for article in data:
            processed_articles.append(preprocess_article(article))
    elif isinstance(data, dict):
        processed_articles.append(preprocess_article(data))
    else:
        print("❌ Unsupported JSON format!")
        return

    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    with open(output_path, "w", encoding="utf-8") as out:
        json.dump(processed_articles, out, ensure_ascii=False, indent=2)

    print(f"✔ All articles saved in single JSON: {output_path}")


# -----------------------------
# Main
# -----------------------------

if __name__ == "__main__":
    INPUT_FILE = r"D:\Menghour\MATES\scraping\notebook\khmer_dual_complete.json"
    OUTPUT_FILE = r"D:\Menghour\MATES\scraping\notebook\khmer_dual_complete_clean.json"

    if os.path.exists(INPUT_FILE):
        process_json_file(INPUT_FILE, OUTPUT_FILE)
    else:
        print("❌ JSON input file not found!")


✔ All articles saved in single JSON: D:\Menghour\MATES\scraping\notebook\khmer_dual_complete_clean.json
